# NMT Benchmarking, End-to-End Reproduction Notebook

**Indicative reference notebook** for reproducing the study. It documents the full workflow in order. Replace `YOUR_GH_TOKEN` and `YOUR_HF_TOKEN` with your own tokens.

Systems Evaluated: Google Translate (baseline), Microsoft Translator, NLLB-200, IndicTrans2 (six Indic languages only), and MADLAD-400.
Corpus: 10,228 English UI strings (OPUS KDE4), 12 target languages.

**Important Environment Notice:** Model families use incompatible dependencies and must be executed in separate runtimes:
- **NLLB-200:** transformers 5.16.1 (GPU)
- **IndicTrans2:** transformers 4.46.1 + IndicTransToolkit (GPU, gated model)
- **MADLAD-400:** FP16 local inference (`requirements-madlad400.txt`)
- **Metrics (COMET/BERTScore/CometKiwi):** Requires `setuptools<81`.

## 1. Clone Repository

In [ ]:
%cd /content
!git clone https://YOUR_GH_TOKEN@github.com/p20220505/nmt-benchmarking-paper.git
%cd nmt-benchmarking-paper
!git config user.email "p20220505@hyderabad.bits-pilani.ac.in"
!git config user.name "Neeraj Kumar Sharma"

## 2. NLLB-200 (Open-source, GPU)

In [ ]:
!pip install -q sentencepiece
!python scripts/run_all_nllb200.py
!pip freeze > requirements-lock.txt

## 3. IndicTrans2 (Open-source, GPU, Gated)

In [ ]:
!pip install -q "transformers==4.46.1" IndicTransToolkit sentencepiece
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")
!python scripts/run_all_indictrans2.py
!pip freeze > requirements-indictrans2.txt

## 4. MADLAD-400 (Open-source, GPU, FP16)

In [ ]:
!pip install -r requirements-madlad400.txt
!python scripts/run_all_madlad400.py

## 5. Microsoft Translator API (Commercial, no GPU needed)

In [ ]:
import os
os.environ["AZURE_TRANSLATOR_KEY"] = "YOUR_AZURE_KEY"
os.environ["AZURE_TRANSLATOR_REGION"] = "eastus"
!pip install -q requests tqdm
!python scripts/run_all_microsoft.py

## 6. Checksums (Data Integrity)

In [ ]:
!for f in data/*/*.txt; do echo "$f: $(wc -l <$f)"; done
!md5sum data/source/en.txt data/nllb200/*.txt data/indictrans2/*.txt data/microsoft_translator/*.txt data/madlad400/*.txt > CHECKSUMS.txt

## 7. Metrics (Lexical, Neural, Reference-free)

In [ ]:
!pip install -q "setuptools<81" sacrebleu "sacrebleu[ja]" nltk rouge-score bert-score unbabel-comet
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")
!python scripts/compute_metrics.py --all --metrics lexical
!python scripts/compute_metrics.py --all --metrics neural
!python scripts/compute_cometkiwi.py --all
!python scripts/merge_metric_results.py
!python scripts/make_pivots.py
!pip freeze > requirements-metrics.txt